# Dentify (CC26-PSU285) – Data Wrangling, Cleaning, Feature Engineering & EDA

This notebook documents the complete Dentify data science workflow: multi-source dental image gathering, quality assessment, manual cleaning, perceptual-hash deduplication, stratified group splitting **before** augmentation, engineered image-quality features, and explanatory charts backing every conclusion.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROCESSED = Path('../data/processed')
CHARTS = Path('../outputs/charts')
assessment = json.loads((PROCESSED / 'assessment_report.json').read_text(encoding='utf-8'))
features_df = pd.read_csv(PROCESSED / 'engineered_features.csv')
print('Assessment summary:', json.dumps(assessment, indent=2))
features_df.head()

## 1. Data Gathering, Cleaning & Anti-Leakage Verification

- Raw files scanned (included sources): **16779**
- Pre-augmented & duplicate YOLO export files removed: **6957**
- Off-target clinical classes dropped (`hypodontia`, `Mouth Ulcer`, `Tooth Discoloration`): **4771**
- Corrupt files dropped: **0**
- Low-res / extreme aspect dropped: **21**
- Cross-dataset exact & perceptual near-duplicates removed (Hamming ≤ 4): **3303**
- Final clean unique images: **1700** (train=1189, val=256, test=255)

**Conclusion 1:** Perceptual hashing eliminated duplicate images across sources before splitting, preventing inflated validation/test accuracy. Every split has balanced class coverage as shown below.

In [ ]:
display(Image(filename=str(CHARTS / '01_class_distribution_by_split.png')))
display(Image(filename=str(CHARTS / '02_source_composition.png')))

## 2. Feature Engineering & Quality-Gate Threshold Justification

- Median mean luminance: **128.2** (5th–95th percentile: 85.7–177.2)
- Median Laplacian sharpness: **321.7**
- Mean CLAHE contrast gain: **4.91**

**Conclusion 2:** Real-world dental photos span wide luminance and sharpness ranges. Setting serving quality-gate thresholds at luminance `[35, 235]` and Laplacian variance `≥ 30` rejects dark/blurry captures without discarding well-lit screening photos, while CLAHE improves local enamel/gingiva contrast consistently across classes.

In [ ]:
display(Image(filename=str(CHARTS / '03_quality_features_by_class.png')))
display(Image(filename=str(CHARTS / '04_clahe_contrast_and_feature_correlation.png')))